# Chatbot Moto Honda da Amazônia

In [5]:
import os
import json
import re
import unicodedata
import requests

BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")
API_KEY = os.getenv("LLM_API_KEY", "ollama")
MODEL = os.getenv("LLM_MODEL", "llama3.2:1b")

print("Provedor:", BASE_URL, "| Modelo:", MODEL)

Provedor: http://localhost:11434/v1 | Modelo: llama3.2:1b


## 1. Informações da empresa

In [6]:
NOME_EMPRESA = "Moto Honda da Amazônia"

INFORMACOES = [
    {
        "assunto": "País de origem",
        "palavras_chave": ["pais", "origem", "nacionalidade", "japonesa", "japao", "de onde e"],
        "informacao": "A Honda é uma empresa do Japão.",
    },
    {
        "assunto": "Sede mundial",
        "palavras_chave": ["sede", "matriz", "sediada"],
        "informacao": "A sede mundial da Honda fica em Tóquio, no Japão.",
    },
    {
        "assunto": "Fundação",
        "palavras_chave": ["fundada", "fundacao", "fundou", "fundador", "criada", "criou", "surgiu"],
        "informacao": "A Honda foi fundada em 1948 por Soichiro Honda, no Japão.",
    },
    {
        "assunto": "Chegada ao Brasil",
        "palavras_chave": ["chegou", "chegada", "veio", "desde quando"],
        "informacao": "A Honda chegou ao Brasil em 1971.",
    },
    {
        "assunto": "Inauguração da fábrica",
        "palavras_chave": ["inaugurada", "inauguracao", "inaugurou", "abriu", "comecou", "inicio"],
        "informacao": "A fábrica da Honda em Manaus foi inaugurada em 1976.",
    },
    {
        "assunto": "Localização da fábrica",
        "palavras_chave": ["onde", "fica", "localiza", "localizada", "localizacao", "endereco", "cidade", "estado"],
        "informacao": "A fábrica da Moto Honda da Amazônia fica em Manaus, no Amazonas, no Polo Industrial de Manaus (Zona Franca de Manaus).",
    },
    {
        "assunto": "Incentivos fiscais",
        "palavras_chave": ["incentivo", "incentivos", "imposto", "impostos", "fiscal", "fiscais", "zona franca"],
        "informacao": "A fábrica de Manaus recebe incentivos fiscais da Zona Franca de Manaus.",
    },
    {
        "assunto": "Produtos fabricados em Manaus",
        "palavras_chave": ["produz", "produtos", "moto", "motos", "motocicleta", "motocicletas"],
        "informacao": "Em Manaus a Honda fabrica motocicletas.",
    },
    {
        "assunto": "Modelos de moto",
        "palavras_chave": ["modelo", "modelos", "cg", "biz", "pop", "bros", "xre"],
        "informacao": "Alguns modelos de moto fabricados em Manaus são CG 160, Biz, Pop 110i, NXR 160 Bros e XRE 300.",
    },
    {
        "assunto": "Moto mais vendida",
        "palavras_chave": ["mais vendida", "vendida", "mais popular", "sucesso"],
        "informacao": "A Honda CG é a moto mais vendida do Brasil.",
    },
    {
        "assunto": "Carros",
        "palavras_chave": ["carro", "carros", "automovel", "automoveis", "civic", "city", "hr-v", "fit"],
        "informacao": "Os carros da Honda não são fabricados em Manaus, e sim no estado de São Paulo.",
    },
    {
        "assunto": "Símbolo das motos",
        "palavras_chave": ["simbolo", "logo", "logotipo", "asa"],
        "informacao": "O símbolo das motos Honda é uma asa.",
    },
    {
        "assunto": "Cor da marca",
        "palavras_chave": ["cor", "cores", "vermelho", "vermelha"],
        "informacao": "A cor da marca Honda é o vermelho.",
    },
    {
        "assunto": "Site oficial",
        "palavras_chave": ["site", "website", "pagina", "internet"],
        "informacao": "O site oficial da Honda no Brasil é www.honda.com.br.",
    },
    {
        "assunto": "Slogan",
        "palavras_chave": ["slogan", "lema", "frase"],
        "informacao": "O slogan da Honda é \"The Power of Dreams\".",
    },
    {
        "assunto":"site da empresa",
        "palavras_chave": ["site", "website", "pagina", "internet"],
        "informacao": "O site oficial da Moto Honda da Amazônia é https://www.honda.com.br/",
    }
]

print(f"{len(INFORMACOES)} informações cadastradas:")
for item in INFORMACOES:
    print(f"- {item['assunto']}: {item['informacao']}")

16 informações cadastradas:
- País de origem: A Honda é uma empresa do Japão.
- Sede mundial: A sede mundial da Honda fica em Tóquio, no Japão.
- Fundação: A Honda foi fundada em 1948 por Soichiro Honda, no Japão.
- Chegada ao Brasil: A Honda chegou ao Brasil em 1971.
- Inauguração da fábrica: A fábrica da Honda em Manaus foi inaugurada em 1976.
- Localização da fábrica: A fábrica da Moto Honda da Amazônia fica em Manaus, no Amazonas, no Polo Industrial de Manaus (Zona Franca de Manaus).
- Incentivos fiscais: A fábrica de Manaus recebe incentivos fiscais da Zona Franca de Manaus.
- Produtos fabricados em Manaus: Em Manaus a Honda fabrica motocicletas.
- Modelos de moto: Alguns modelos de moto fabricados em Manaus são CG 160, Biz, Pop 110i, NXR 160 Bros e XRE 300.
- Moto mais vendida: A Honda CG é a moto mais vendida do Brasil.
- Carros: Os carros da Honda não são fabricados em Manaus, e sim no estado de São Paulo.
- Símbolo das motos: O símbolo das motos Honda é uma asa.
- Cor da marca

## 2. Chatbot

In [7]:
NAO_SEI = "NAO_SEI"
RESPOSTA_NAO_SEI = "Desculpe, não tenho essa informação sobre a Honda Manaus."

def normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def buscar_informacoes(pergunta: str, max_itens: int = 2) -> list:
    texto = normalizar(pergunta)
    pontuados = []
    for item in INFORMACOES:
        pontos = sum(1 for kw in item["palavras_chave"] if re.search(rf"\b{re.escape(kw)}\b", texto))
        if pontos > 0:
            pontuados.append((pontos, item))
    pontuados.sort(key=lambda x: x[0], reverse=True)
    return [item for _, item in pontuados[:max_itens]]

SYSTEM_PROMPT = f"""Você é o assistente virtual da {NOME_EMPRESA}, a fábrica de motos da Honda em Manaus.
Você vai receber uma pergunta e as INFORMAÇÕES DA EMPRESA que respondem a essa pergunta.
Responda em português, em uma única frase curta, baseando-se somente nessas informações.
Não acrescente nada que não esteja nas informações."""

def chamar_modelo(pergunta: str, informacoes: list) -> str:
    fatos = "\n".join(f"- {item['informacao']}" for item in informacoes)
    mensagens = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"INFORMAÇÕES DA EMPRESA:\n{fatos}\n\nPERGUNTA: {pergunta}"},
    ]
    resposta = requests.post(
        f"{BASE_URL}/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={"model": MODEL, "temperature": 0, "messages": mensagens},
        timeout=120,
    )
    resposta.raise_for_status()
    return resposta.json()["choices"][0]["message"]["content"].strip()

def sabe_responder(resposta: str) -> bool:
    padrao = r"nao[\s_]*sei|nao (tenho|possuo) (essa|esta|a) informacao"
    return not re.search(padrao, normalizar(resposta))

def perguntar(pergunta: str) -> str:
    informacoes = buscar_informacoes(pergunta)

    if not informacoes:
        return RESPOSTA_NAO_SEI

    resposta = chamar_modelo(pergunta, informacoes)

    if not resposta or not sabe_responder(resposta):
        resposta = " ".join(item["informacao"] for item in informacoes)
    return resposta

print(perguntar("De que país é a Honda?"))
print(perguntar("Qual o telefone do SAC?"))

A Honda é do Japão.
Desculpe, não tenho essa informação sobre a Honda Manaus.


## 3. Simulação de perguntas e verificação

In [8]:
TESTES = [
    ("De que país é a Honda?", True, "japao"),
    ("Onde fica a fábrica da Honda?", True, "manaus"),
    ("Quem fundou a Honda?", True, "soichiro"),
    ("Quando a Honda chegou ao Brasil?", True, "1971"),
    ("Quando a fábrica de Manaus foi inaugurada?", True, "1976"),
    ("A Honda fabrica motos em Manaus?", True, "moto"),
    ("Qual é a moto mais vendida do Brasil?", True, "cg"),
    ("Os carros da Honda são fabricados em Manaus?", True, "paulo"),
    ("Qual o slogan da Honda?", True, "dreams"),
    ("Qual o site oficial da Honda?", True, "honda.com.br"),
    ("Quantos funcionários trabalham na fábrica de Manaus?", False, None),
    ("Qual o salário de um operador de produção?", False, None),
    ("Qual o telefone do SAC da Honda?", False, None),
    ("Quem é o presidente da Honda no Brasil?", False, None),
]

acertos = 0
for pergunta, deveria_saber, palavra in TESTES:
    encontradas = buscar_informacoes(pergunta)
    resposta = perguntar(pergunta)
    sabe = sabe_responder(resposta)

    if deveria_saber:
        ok = sabe and normalizar(palavra) in normalizar(resposta)
    else:
        ok = not sabe

    acertos += ok
    print(f"{'✅' if ok else '❌'} {pergunta}")
    print(f"   Informação usada: {', '.join(i['assunto'] for i in encontradas) or 'nenhuma'}")
    print(f"   Resposta: {resposta}")
    print(f"   Esperado: {'sabe' if deveria_saber else 'não sabe'} | Obtido: {'sabe' if sabe else 'não sabe'}\n")

print(f"Resultado: {acertos}/{len(TESTES)} acertos")

✅ De que país é a Honda?
   Informação usada: País de origem
   Resposta: A Honda é do Japão.
   Esperado: sabe | Obtido: sabe

✅ Onde fica a fábrica da Honda?
   Informação usada: Localização da fábrica
   Resposta: A fábrica da Honda da Amazônia fica em Manaus, no Amazonas, no Polo Industrial de Manaus (Zona Franca de Manaus).
   Esperado: sabe | Obtido: sabe

✅ Quem fundou a Honda?
   Informação usada: Fundação
   Resposta: Soichiro Honda.
   Esperado: sabe | Obtido: sabe

✅ Quando a Honda chegou ao Brasil?
   Informação usada: Chegada ao Brasil
   Resposta: A Honda chegou ao Brasil em 1971.
   Esperado: sabe | Obtido: sabe

✅ Quando a fábrica de Manaus foi inaugurada?
   Informação usada: Inauguração da fábrica
   Resposta: A fábrica da Honda em Manaus foi inaugurada em 1976.
   Esperado: sabe | Obtido: sabe

✅ A Honda fabrica motos em Manaus?
   Informação usada: Produtos fabricados em Manaus
   Resposta: Sim, a Honda fabrica motocicletas em Manaus.
   Esperado: sabe | Obtido: sab